# Lesson 2. Encoder comparison: does a newer model win?

Same studies, same slices, same head, same CV. Change **only the frozen encoder**
and compare. That is how you connect a change in score to the one thing you changed.

The encoders:

| Encoder | What it is | Link |
|---|---|---|
| ResNet-18 | supervised on ImageNet (natural images) | torchvision |
| DINOv2 | self-supervised on 142M natural images | [`facebook/dinov2-base`](https://huggingface.co/facebook/dinov2-base) |
| DINOv3 | newer self-supervised foundation model (gated) | [`facebook/dinov3-vitb16-pretrain-lvd1689m`](https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m) |
| MedImageInsight | medical image and text encoder (Microsoft) | [Azure catalog](https://ai.azure.com/catalog/models/MedImageInsight) · [community mirror](https://huggingface.co/lion-ai/MedImageInsights) |

You run the first two here. DINOv3 and MedImageInsight are guided exercises with
hooks at the end.

**Where you are: Lesson 2 of 3, the main result.** You run two encoders on the same
data. Then you answer the key question of the course: can you tell which encoder is
better from a class-sized sample? The two experiments and the confidence-interval
calculation below give the answer.

Go deeper: [DINOv2 paper](https://arxiv.org/abs/2304.07193),
[why linear probing measures a frozen encoder](https://cs231n.github.io/transfer-learning/).

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# config: read a bounded sample, never the full ~570 GB ----------------------
DATA_ROOT = os.environ.get("RSNA_DATA_ROOT",
    "/kaggle/input/rsna-knee-abnormality-detection")  # competition mount
N_STUDIES = 300     # read only this many studies' DICOMs, not the full dataset
K_SLICES  = 5       # central slices per study
SEED      = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def _find_file(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isfile(p):
            return p
    for n in names:
        hits = sorted(glob.glob(os.path.join(root, "**", n), recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"none of {names} under {root}")

def _find_dir(root, name):
    p = os.path.join(root, name)
    if os.path.isdir(p):
        return p
    hits = sorted(d for d in glob.glob(os.path.join(root, "**", name), recursive=True)
                  if os.path.isdir(d))
    if not hits:
        raise FileNotFoundError(f"dir {name} not found under {root}")
    return hits[0]

TRAIN_CSV        = _find_file(DATA_ROOT, ["train.csv", "metadata/train.csv"])
SERIES_CSV       = _find_file(DATA_ROOT, ["train_series.csv", "metadata/train_series.csv"])
TRAIN_SERIES_DIR = _find_dir(DATA_ROOT, "train_series")
print("labels :", TRAIN_CSV)
print("series :", SERIES_CSV)
print("dicoms :", TRAIN_SERIES_DIR)

In [ ]:
import pydicom

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def list_studies(train_series_dir, n=None):
    studies = sorted(d for d in os.listdir(train_series_dir)
                     if os.path.isdir(os.path.join(train_series_dir, d)))
    return studies[:n] if n else studies

def first_sagittal_series(series_df, study, train_series_dir):
    study_dir = os.path.join(train_series_dir, study)
    rows = series_df[(series_df["StudyInstanceUID"] == study) &
                     (series_df["Anatomical_Plane"].astype(str).str.lower() == "sagittal")]
    for sid in rows["SeriesInstanceUID"].astype(str):
        sdir = os.path.join(study_dir, sid)
        if os.path.isdir(sdir) and glob.glob(os.path.join(sdir, "*.dcm")):
            return sdir
    for sdir in sorted(glob.glob(os.path.join(study_dir, "*"))):
        if glob.glob(os.path.join(sdir, "*.dcm")):
            return sdir
    return None

def _decode(ds):
    px = np.asarray(ds.pixel_array, dtype=np.float32)
    px = px * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")) == "MONOCHROME1":
        px = float(px.max() + px.min()) - px
    return px

def load_central_slices(series_dir, k=K_SLICES):
    recs = []
    for i, p in enumerate(sorted(glob.glob(os.path.join(series_dir, "*.dcm")))):
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            continue
        try:
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            coord = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            coord = float(i)
        try:
            px = _decode(ds)
        except Exception:
            continue
        if px.ndim == 2:
            recs.append((coord, px))
    if not recs:
        return []
    recs.sort(key=lambda r: r[0])
    lo = max(0, len(recs) // 2 - k // 2)
    return [px for _, px in recs[lo:lo + k]]

def _norm01(img):
    finite = img[np.isfinite(img)]
    if finite.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    clean = np.nan_to_num(img, nan=float(lo), posinf=float(hi), neginf=float(lo))
    return np.clip((clean - lo) / (hi - lo), 0, 1).astype(np.float32)

def prep_batch(slices, size=224):
    frames = []
    for s in slices:
        t = torch.from_numpy(_norm01(s))[None, None]
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        frames.append(t[0, 0])
    x = torch.stack(frames)[:, None].repeat(1, 3, 1, 1)
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x - mean) / std

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

def cv_auc(X, y, seed=SEED, n_splits=5, shuffle_labels=False):
    """Out-of-fold ROC-AUC. Each study is one row, so folds never split a study."""
    y = np.asarray(y).astype(int)
    if shuffle_labels:
        y = y[np.random.default_rng(seed).permutation(len(y))]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr, va in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=1000).fit(sc.transform(X[tr]), y[tr])
        oof[va] = clf.predict_proba(sc.transform(X[va]))[:, 1]
    return float(roc_auc_score(y, oof))

def mean_auc(X, y, shuffle_labels=False, seeds=range(10)):
    """Average AUC over several CV seeds -- stable when N is small and noisy."""
    return float(np.mean([cv_auc(X, y, seed=s, shuffle_labels=shuffle_labels)
                          for s in seeds]))

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

def build_resnet18():
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = torch.nn.Identity()          # 512-d penultimate features
    return m.eval().to(device)

@torch.inference_mode()
def resnet18_features(model, batch):
    return model(batch.to(device)).float().cpu()   # [k, 512]

def study_feature(model, feats_fn, slices):
    return feats_fn(model, prep_batch(slices)).mean(0).numpy()  # mean-pool slices

## Step 1: DINOv2 as a drop-in encoder

Transformers ships DINOv2. Its features are the patch tokens, and a good
linear-probe feature is the mean of the patch tokens with the CLS token dropped.
The same `study_feature` mean-pools over slices afterward.

In [ ]:
from transformers import AutoModel

def build_dinov2():
    return AutoModel.from_pretrained("facebook/dinov2-base").eval().to(device)

@torch.inference_mode()
def dinov2_features(model, batch):
    out = model(pixel_values=batch.to(device)).last_hidden_state.float().cpu()
    return out[:, 1:].mean(1)   # mean of patch tokens (drop CLS at index 0) -> [k, 768]

## Step 2: Run the comparison

Extract the features once per encoder and keep them, so the experiments below can
re-score without a repeat of the feature extraction.

In [ ]:
labels = pd.read_csv(TRAIN_CSV); series = pd.read_csv(SERIES_CSV)
lab = labels.set_index("StudyInstanceUID")
studies = list_studies(TRAIN_SERIES_DIR, n=N_STUDIES)

def features_over_studies(model, feats_fn):
    X, kept = [], []
    for s in studies:
        sd = first_sagittal_series(series, s, TRAIN_SERIES_DIR)
        if sd is None:
            continue
        sl = load_central_slices(sd)
        if not sl:
            continue
        X.append(study_feature(model, feats_fn, sl)); kept.append(s)
    return np.stack(X), kept

feats = {}   # name -> (X, kept), reused by the experiments below
for name, build, fn in [("resnet18", build_resnet18, resnet18_features),
                        ("dinov2",   build_dinov2,   dinov2_features)]:
    m = build()
    feats[name] = features_over_studies(m, fn)
    del m; torch.cuda.empty_cache()

rows = []
for name, (X, kept) in feats.items():
    for tgt in ["Effusion", "ACL"]:
        y = lab.loc[kept, tgt].to_numpy()
        rows.append((name, tgt, round(cv_auc(X, y), 3),
                     round(cv_auc(X, y, shuffle_labels=True), 3)))
print(pd.DataFrame(rows, columns=["encoder","target","auc","shuffle_auc"]).to_string(index=False))

## Step 3: What you see, and the risk in it

On a class-sized sample the two encoders are close, and the winner can change when
you change `SEED` or `N_STUDIES`. It is easy to write "encoder X beats Y." Do not,
at least not from this N.

Here is the same comparison at two sample sizes (ACL):

| ACL | shuffle | ResNet-18 | DINOv2 |
|---|---|---|---|
| **N = 917** (KneeMRI, reliable) | 0.49 | **0.715** | **0.707** |
| N = 58 (tiny) | 0.54 | 0.67 | 0.53 |

*(All columns are 5-seed means. The live cells use one seed, so your numbers are
near these, not equal to them.)*

At N=58 ResNet-18 is much higher. At N=917 the gap is gone: the two encoders are
equal, and both score above the shuffle control. The N=58 lead was noise. The lesson
is not "ResNet beats DINOv2." It is that you cannot rank encoders on a small sample,
and that no natural-image encoder scores higher on knees. That is the reason to test
a medical encoder, which is Exercise 2.

## Experiment: change the seed, watch the scores move

Re-score both encoders on ACL across six seeds. The features do not change; only the
fold split changes. Watch two results: how far each encoder AUC changes as you move
the seed, and whether the ranking between them holds. If the ranking changes with the
seed, the difference was never real. Even when one encoder wins every seed, the next
cell checks whether that lead is larger than the error bar.

In [ ]:
Xr, kr = feats["resnet18"]
Xd, kd = feats["dinov2"]
yr = lab.loc[kr, "ACL"].to_numpy()
yd = lab.loc[kd, "ACL"].to_numpy()
for seed in range(6):
    a = cv_auc(Xr, yr, seed=seed); b = cv_auc(Xd, yd, seed=seed)
    print(f"seed={seed}  resnet18={a:.3f}  dinov2={b:.3f}  winner={'resnet18' if a>b else 'dinov2'}")

## Step 4: How sure are we? Put an error bar on the AUC

The seed changes show the noise. Now measure it. An AUC from a few dozen patients is
an estimate with a standard error. Hanley and McNeil (1982) give that error in
closed form, from the AUC and the positive and negative counts. Compute the 95%
interval for your live ACL result and for the N=917 run. The lesson becomes one
number, the width of the interval.

Go deeper: [Hanley & McNeil, Radiology 1982](https://pubmed.ncbi.nlm.nih.gov/7063747/).

In [ ]:
def auc_ci(auc, n_pos, n_neg, z=1.96):
    """Hanley-McNeil 95% CI for an AUC from positive/negative counts."""
    A = auc
    Q1 = A / (2 - A)
    Q2 = 2 * A * A / (1 + A)
    var = (A * (1 - A) + (n_pos - 1) * (Q1 - A * A) + (n_neg - 1) * (Q2 - A * A)) / (n_pos * n_neg)
    se = var ** 0.5
    return se, A - z * se, A + z * se

# both encoders on your live class-sized run (ACL)
for enc, X, kept in [("resnet18", Xr, kr), ("dinov2", Xd, kd)]:
    y = lab.loc[kept, "ACL"].to_numpy()
    a = cv_auc(X, y)
    se, lo, hi = auc_ci(a, int(y.sum()), int(len(y) - y.sum()))
    print(f"{enc:9s} ACL  N={len(y)}  AUC={a:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (+/-{1.96*se:.3f})")
print("Those two intervals overlap by a large amount. Whichever encoder won your run,")
print("the lead is not significant at this N; the error bars do not separate.\n")

# the large-N reference run (ACL, KneeMRI): 227 positives, 690 negatives
se2, lo2, hi2 = auc_ci(0.715, 227, 690)
print(f"reference N=917:  AUC=0.715  95% CI [{lo2:.3f}, {hi2:.3f}]  (+/-{1.96*se2:.3f})")
print("At N=917 the interval is smaller, about +/-0.04. That is why the N=917")
print("comparison is reliable and yours is not.")

## Exercise 1: DINOv3 (gated)

DINOv3 is newer and stronger on natural-image benchmarks. Accept the license on the
[model card](https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m),
add your HF token to the kernel, and complete the encoder. Watch the tokens: DINOv3
adds 1 CLS and **4 register tokens** at the front, so the patch tokens start at index
**5**, not 1. Then add it to the comparison and run the confidence-interval cell
again. Does its interval separate from ResNet-18, or overlap?

In [ ]:
# TODO (Exercise 1): implement DINOv3 features, add it to the comparison loop above.
# from transformers import AutoModel
# def build_dinov3():
#     return AutoModel.from_pretrained(
#         "facebook/dinov3-vitb16-pretrain-lvd1689m", token=YOUR_HF_TOKEN).eval().to(device)
# @torch.inference_mode()
# def dinov3_features(model, batch):
#     out = model(pixel_values=batch.to(device)).last_hidden_state.float().cpu()
#     return out[:, 5:].mean(1)   # drop CLS(0) + 4 register tokens -> patch mean

## Exercise 2: MedImageInsight (the medical encoder)

MedImageInsight is trained on medical images and paired text. Unlike the
natural-image encoders, it has seen MRIs. The real pipeline in this repo is built on
it. It is not a one-line `transformers` load. You fetch the weights from the
[Azure catalog](https://ai.azure.com/catalog/models/MedImageInsight) or the
[community mirror](https://huggingface.co/lion-ai/MedImageInsights) and add them as
a **Kaggle dataset**. This is more access friction, the kind Lesson 0 described.
Wrap it to return one vector per slice, then reuse `study_feature` and the same
comparison loop.

The question to answer: does a domain-matched encoder produce a gap over ImageNet
and DINOv2 that is larger than the confidence interval, at an N large enough to
trust?

In [ ]:
# TODO (Exercise 2): load MedImageInsight, expose feats_fn(model, batch) -> [k, D],
# and add ("mii", build_mii, mii_features) to the comparison loop.
# Reference loader in this project: src/rsna_knee/medimageinsight.py

## Recap

To hold the data fixed and swap the encoder is the right experiment, but it is only
useful when N is large enough. On class-sized data the encoders are equal, and the
confidence interval tells you why. The real question, whether a medical encoder
wins, is Exercise 2, and it needs a real sample size to answer. That is the step
toward the advanced pipeline in this repo.